# Tic em Trilhas - Construção de Agentes Inteligentes com IA
## M1A4 — Agentes de Pesquisa com LangGraph + Tavily
Nesta aula montamos um agente ReAct orientado a buscas, combinando LangGraph, ChatGPT e a ferramenta Tavily para responder perguntas que exigem informações atualizadas.

### Inicialização do ambiente e bibliotecas
Carregamos variáveis de ambiente com `dotenv` e importamos LangChain/LangGraph. Esses módulos dão suporte ao encadeamento de estados e à criação do agente que combinará LLM com ferramentas externas (como o mecanismo de busca Tavily).

In [3]:
from dotenv import load_dotenv
import os
_ = load_dotenv()
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

### Registrando a ferramenta de busca
Aqui conectamos o Tavily ao agente. Ele funciona como “olhos na web”, permitindo que o modelo consulte fontes atualizadas quando precisar responder perguntas que exigem fatos recentes.

In [4]:
tool = TavilySearchResults(max_results=3, tavily_api_key=os.getenv("TAVILY_SEARCH_API"))

/var/folders/2g/rcypq52x3y1b53qt3sszm6r00000gn/T/ipykernel_55729/47839059.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=3, tavily_api_key=os.getenv("TAVILY_SEARCH_API"))


### Estado compartilhado entre nós do grafo
O `AgentState` encapsula o histórico de mensagens. O uso de `Annotated[..., operator.add]` indica ao LangGraph que novas mensagens são concatenadas automaticamente a cada iteração.

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


### Montando o agente com LangGraph
Construímos um grafo com dois nós principais:
1. `llm`: consulta o modelo para decidir o próximo passo.
2. `action`: executa as ferramentas solicitadas via `tool_calls`.
As arestas condicionais garantem o ciclo “modelo → ferramenta → modelo” até que não haja mais ações pendentes.

In [6]:
class Agent:

    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Chamando: {t}")
            if not t['name'] in self.tools:      # verificar nome de ferramenta incorreto do LLM
                print("\n ....nome de ferramenta incorreto....")
                result = "nome de ferramenta incorreto, tente novamente"  # instruir LLM a tentar novamente
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("De volta ao modelo!")
        return {'messages': results}

### Prompt de sistema focado em pesquisa
O texto abaixo orienta o LLM a agir como pesquisador disciplinado: só buscar quando fizer sentido, explicar múltiplas buscas e usar o motor como apoio antes de responder.

In [7]:
prompt = """Você é um assistente de pesquisa inteligente. Use o motor de busca para procurar informações. \
Você pode fazer múltiplas chamadas (juntas ou em sequência). \
Só procure informações quando tiver certeza do que quer. \
Se precisar procurar algumas informações antes de fazer uma pergunta de acompanhamento, você pode fazer isso!
"""

### Instanciando o modelo com ferramentas ligadas
Enlaçamos o `ChatOpenAI` ao grafo e passamos a lista de ferramentas disponíveis. A partir daqui, cada mensagem do humano percorre automaticamente o fluxo definido.

In [8]:
model = ChatOpenAI(model="gpt-4o-mini")
abot = Agent(model, [tool], system=prompt)

### Exemplo 1 — Pergunta única com uma chamada de ferramenta
Testamos o agente com uma pergunta meteorológica. Observe no output como o LLM pede o Tavily, aguarda o `ToolMessage` e só então compõe a resposta final para o usuário.

In [10]:
messages = [HumanMessage(content="Qual é o clima no Rio de Janeiro?")]
result = abot.graph.invoke({"messages": messages})
print("Resultado completo:")
print(result)
print("\nResposta final:")
print(result['messages'][-1].content)

Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'clima no Rio de Janeiro'}, 'id': 'call_s7OuVrO9tQHki5kL3cvB9arZ', 'type': 'tool_call'}
De volta ao modelo!
Resultado completo:
{'messages': [HumanMessage(content='Qual é o clima no Rio de Janeiro?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_s7OuVrO9tQHki5kL3cvB9arZ', 'function': {'arguments': '{"query":"clima no Rio de Janeiro"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 148, 'total_tokens': 171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-ByOZB4973FytFHdePtxrEVOaxP1Lx', 'service_tie

### Exemplo 2 — Perguntas encadeadas e múltiplas buscas
Agora o agente precisa combinar dois fatos (campeão da Copa de 2022 + PIB do país). O grafo permite múltiplas `tool_calls` em sequência, mostrando como o ReAct se generaliza para fluxos de raciocínio mais longos.

In [11]:
query = "Quem ganhou a Copa do Mundo de 2022? Qual é o PIB desse país? Responda cada pergunta." 
messages = [HumanMessage(content=query)]
abot = Agent(model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)

Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'Copa do Mundo de 2022 vencedor'}, 'id': 'call_EjqBQWpuFngeZ6wvdsOTXCLW', 'type': 'tool_call'}
Chamando: {'name': 'tavily_search_results_json', 'args': {'query': 'PIB Argentina 2023'}, 'id': 'call_RBcMu6iujp58kI7HPmofWdKJ', 'type': 'tool_call'}
De volta ao modelo!
A Copa do Mundo de 2022 foi vencida pela **Argentina**. A final ocorreu no Estádio Nacional de Lusail, no Catar, em 18 de dezembro de 2022, onde a Argentina derrotou a França nos pênaltis após um empate de 3 a 3 no tempo regulamentar e na prorrogação. Essa vitória marcou o terceiro título mundial da Argentina, que já havia conquistado a Copa em 1978 e 1986. [Veja mais aqui](https://www.bbc.com/portuguese/internacional-64018691).

Em relação ao PIB da Argentina, em 2023, a economia do país teve um Produto Interno Bruto (PIB) de aproximadamente **596,82 bilhões de euros** (ou 646,08 bilhões de dólares). Este valor representa uma queda de 1,6% em relação ao PIB d